In [11]:
from google import genai
from google.genai import types
from PIL import Image

In [12]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="us-central1",
)

In [20]:
# 2. Open your local image file

paths = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/pink.png","test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"]
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

7


In [14]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

<class 'str'>


## Result logging

Every model call below is logged to `notebook_results/results_log.jsonl` via `result_logger.log_result()`.
This keeps a permanent record across batches without cluttering this notebook.

**For each new batch of images:**
1. Bump `BATCH_ID` below (e.g. `"batch2"`)
2. Update `paths` in the image-loading cell to point at the new images
3. Re-run the experiment cells as usual — results from previous batches stay in the log

Use the "Review logged results" section at the end to compare across batches.

In [15]:
import time
from result_logger import log_result, load_results

BATCH_ID = "batch1"  # bump this for each new set of test images

## Try the first batch with the same model for now to test the workflow

In [23]:
image_list = list(zip(paths, images))  

for image_path, image in image_list:

    start = time.perf_counter()

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.05  # To change to variable this later
        )
    )

    latency_s = round(time.perf_counter() - start, 2)

    # 4. Print the text result
    print(response.text)

    log_result(
        batch_id=BATCH_ID,
        image_path=image_path,
        prompt_id="V1",
        model="gemini-2.5-flash",
        temperature=0.05,
        raw_response=response.text,
        latency_s=latency_s,
    )

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document displays a high degree of visual consistency across all text elements. The font style, weight, sharpness, and resolution are uniform throughout the entire document, from the title to the address and date. There are no discernible variations in anti-aliasing, pixelation, or character rendering quality. The black text maintains a consistent color and contrast against the white background, with no halos, outlines, or discontinuities around specific characters or words. Baselines are consistent, and spacing appears natural. All visual signals indicate that the text was generated as a single, cohesive digital image rather than being altered post-recapture.",
  "signals_detected": [],
  "notes": "The document is a synthetic, digitally generated image, and its authenticity refers to the absence of post-recapture digital alteration, not its real-world validity."
}
```
```json
{
  "verdict": "authentic",
  

In [24]:
# V1 split: persona passed as system_instruction, task prompt as the user turn

image_list = list(zip(paths, images))  # image 1 and 2

for image_path, image in image_list:
    start = time.perf_counter()

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )

    latency_s = round(time.perf_counter() - start, 2)

    print(response.text)

    log_result(
        batch_id=BATCH_ID,
        image_path=image_path,
        prompt_id="V1_split",
        model="gemini-2.5-flash",
        temperature=0.1,
        raw_response=response.text,
        latency_s=latency_s,
    )

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document exhibits a consistent typography baseline across all text regions. All characters, including Thai script and numbers, maintain uniform font weight, sharpness, resolution, and size. There are no visible inconsistencies in shadows, edges, backgrounds, color, or contrast around specific text areas. The overall rendering quality is uniform, suggesting the text was generated digitally as a single entity rather than having parts altered or inserted after recapture.",
  "signals_detected": [],
  "notes": "The document appears to be a clean, digitally generated image with no visual artifacts indicative of tampering."
}
```
```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document exhibits a consistent typography baseline across all text regions. The font weight, sharpness, and resolution are uniform throughout. There are no discernible font size mismatches, color or contrast 

## Review logged results

Load the full log (all batches) or just the current `BATCH_ID` to compare prompts/models without re-running anything.

In [25]:
df_all = load_results()
df_all[["timestamp", "batch_id", "image_path", "prompt_id", "model", "latency_s"]]

,timestamp,batch_id,image_path,prompt_id,model,latency_s
0,2026-06-11T02:20:36.332953+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,9.42
1,2026-06-11T02:20:47.181659+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,10.85
2,2026-06-11T02:20:53.951337+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-flash,6.76
3,2026-06-11T02:21:04.694144+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-flash,10.74
4,2026-06-11T02:24:43.831825+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,9.18
5,2026-06-11T02:24:51.311197+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,7.48
6,2026-06-11T02:25:08.611948+00:00,batch1,test_images/combined.png,V1,gemini-2.5-flash,17.30
7,2026-06-11T02:25:16.513412+00:00,batch1,test_images/neat.png,V1,gemini-2.5-flash,7.90
8,2026-06-11T02:25:28.105398+00:00,batch1,test_images/neatest.png,V1,gemini-2.5-flash,11.59
9,2026-06-11T02:25:42.508209+00:00,batch1,test_images/cropped-obvious.png,V1,gemini-2.5-flash,14.40


In [32]:
df_all.iloc[0]['parsed_response']

{'verdict': 'authentic',
 'confidence': 'high',
 'reasoning': 'The document displays consistent typography across all text regions. The font style, weight, sharpness, and resolution are uniform. There are no visible inconsistencies in font size, except for the title being slightly larger, which is a standard formatting practice. Edges of characters are clean, and there are no signs of mismatched shadows, color discontinuities, or background anomalies around any specific text. The overall visual integrity of the text suggests it was generated as a single, cohesive digital document without post-recapture alterations.',
 'signals_detected': [],
 'notes': 'The document is digitally generated, and all text elements appear to be part of the original generation.'}

In [22]:
def extract_verdict(parsed: dict | None) -> str | None:
    if isinstance(parsed, dict):
        return parsed.get("verdict")
    return None

current_batch = load_results(batch_id=BATCH_ID).copy()
current_batch["verdict"] = current_batch["parsed_response"].apply(extract_verdict)
current_batch[["image_path", "prompt_id", "verdict", "latency_s"]]

,image_path,prompt_id,verdict,latency_s
0,test_images/First_Tamper.png,V1,authentic,9.42
1,test_images/first-tamper.png,V1,tampered,10.85
2,test_images/First_Tamper.png,V1_split,authentic,6.76
3,test_images/first-tamper.png,V1_split,authentic,10.74
4,test_images/First_Tamper.png,V1,authentic,9.18
5,test_images/first-tamper.png,V1,authentic,7.48
6,test_images/combined.png,V1,tampered,17.30
7,test_images/neat.png,V1,authentic,7.90
8,test_images/neatest.png,V1,tampered,11.59
9,test_images/cropped-obvious.png,V1,authentic,14.40


In [11]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      avg_logprobs=-1.2320455575918223,
      content=Content(
        parts=[
          Part(
            text="""```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document's typography, including font weight, sharpness, resolution, and color, is highly consistent across all text regions. Specifically, the numerical value '๖๐๐,๐๐๐' (600,000) matches the surrounding Thai text in every visual aspect. There are no signs of mismatched shadows, edges, backgrounds, or color/contrast discontinuities around any specific characters or numbers. All text appears to be rendered uniformly, suggesting it is part of the original digital generation rather than a post-recapture alteration.",
  "signals_detected": [],
  "notes": ""
}
```"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>
    ),
  ],
  create_time=datetime.dat